# 🚀 Employment Prediction Hackathon — Python Starter Kit
Welcome! This notebook takes you from raw data to your first leaderboard submission, step by step.

**The goal:** for each person in Round 9, predict the *probability* that they are **employed (1)** vs **unemployed (0)**.
Submissions are scored on **AUC**, so always submit a probability between 0 and 1 — never a hard 0/1.

## 🛠️ Step 1: Load tools and data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
PAL = {'Unemployed':'#E15759','Employed':'#59A14F'}

data_dir = '/kaggle/input/competitions/predict-the-labour-market-status-of-participants'
train_data = pd.read_csv(f'{data_dir}/train.csv', low_memory=False)
test_data  = pd.read_csv(f'{data_dir}/test.csv',  low_memory=False)
print(f'Training rows: {len(train_data)} | columns: {train_data.shape[1]}')
print(f'Testing rows : {len(test_data)} | columns: {test_data.shape[1]}')
# Target in train_data is `employed_status` (1=employed, 0=unemployed); absent from test_data.


## 📊 Step 2: Get to know the data
Each chart answers one question a first-time viewer should ask, and ends with **what it means for your model**.

In [ ]:
plot_df = train_data[train_data['employed_status'].notna()].copy()
plot_df['Status'] = plot_df['employed_status'].map({0:'Unemployed',1:'Employed'})
emp_rate = plot_df['employed_status'].mean()
print(f'Overall employment rate: {emp_rate:.1%}')


### 2.1 — How many people are employed? (class balance)

In [ ]:
counts = plot_df['Status'].value_counts()
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(counts.index, counts.values, color=[PAL[s] for s in counts.index], edgecolor='grey')
for i,v in enumerate(counts.values): ax.text(i, v, f'{v:,}', ha='center', va='bottom')
ax.set_title(f'Most people are unemployed  (employment rate: {emp_rate:.1%})', fontweight='bold')
ax.set_ylabel('People'); ax.margins(y=0.15); plt.tight_layout(); plt.show()


**What it means:** classes are imbalanced (~1 in 3 employed). Predicting 'unemployed' for everyone scores ~68% accuracy yet is useless — which is why the metric is **AUC**, rewarding how well you *rank* people by employment probability.

### 2.2 — Who has history, and who is brand new?

In [ ]:
miss = train_data.isna().mean().sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(miss.index[::-1], miss.values[::-1], color='#4E79A7', edgecolor='grey')
ax.set_title('Which columns are mostly empty?', fontweight='bold')
ax.set_xlabel('Share missing'); ax.xaxis.set_major_formatter(lambda x,_: f'{x:.0%}')
plt.tight_layout(); plt.show()


In [ ]:
hist = train_data['total_historical_rounds'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(hist.index.astype(str), hist.values, color='#8CD17D', edgecolor='grey')
ax.set_title('Most people are first-timers', fontweight='bold')
ax.set_xlabel('Total historical rounds'); ax.set_ylabel('People'); plt.tight_layout(); plt.show()


**What it means:** lag features are powerful when present but empty for most people. Treat 'missing' deliberately rather than dropping rows.

### 2.3 — Does last round's status predict this round?

In [ ]:
hd = plot_df[plot_df['status_broad_lag'].notna()]
rate_by = hd.groupby('status_broad_lag')['employed_status'].mean().sort_values()
fig, ax = plt.subplots(figsize=(8,4.5))
ax.barh(rate_by.index, rate_by.values, color='#59A14F', edgecolor='grey')
for i,v in enumerate(rate_by.values): ax.text(v, i, f' {v:.0%}', va='center')
ax.set_title('Past status shapes the present', fontweight='bold')
ax.set_xlabel('Share now employed'); ax.xaxis.set_major_formatter(lambda x,_: f'{x:.0%}'); ax.margins(x=0.12)
plt.tight_layout(); plt.show()


**What it means:** past status carries signal, but turning it into leaderboard gains takes care — it is not the whole story.

### 2.4 — The gender gap

In [ ]:
g = plot_df[plot_df['gender'].notna()].groupby('gender')['employed_status'].mean()
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(g.index, g.values, color='#4E79A7', edgecolor='grey', width=0.6)
for i,v in enumerate(g.values): ax.text(i, v, f'{v:.1%}', ha='center', va='bottom')
ax.set_title('Employment rate differs by gender', fontweight='bold')
ax.set_ylabel('Employment rate'); ax.yaxis.set_major_formatter(lambda x,_: f'{x:.0%}'); ax.margins(y=0.15)
plt.tight_layout(); plt.show()


### 2.5 — Does education move the needle?

In [ ]:
e = plot_df[plot_df['education_level'].notna()].groupby('education_level')['employed_status'].agg(['mean','count'])
e = e[e['count']>=50].sort_values('mean')
fig, ax = plt.subplots(figsize=(9,5))
ax.barh(e.index, e['mean'], color='#B07AA1', edgecolor='grey')
for i,v in enumerate(e['mean']): ax.text(v, i, f' {v:.0%}', va='center', fontsize=9)
ax.set_title('Employment rate by education level (groups >= 50)', fontweight='bold')
ax.set_xlabel('Employment rate'); ax.xaxis.set_major_formatter(lambda x,_: f'{x:.0%}'); ax.margins(x=0.15)
ax.tick_params(axis='y', labelsize=8); plt.tight_layout(); plt.show()


### 2.6 — Does the Work Readiness Score matter?

In [ ]:
sd = plot_df[plot_df['work_readiness_score'].notna()]
groups = [sd.loc[sd['Status']==s,'work_readiness_score'] for s in ['Unemployed','Employed']]
fig, ax = plt.subplots(figsize=(6,4))
bp = ax.boxplot(groups, tick_labels=['Unemployed','Employed'], patch_artist=True, showfliers=False)
for patch,s in zip(bp['boxes'], ['Unemployed','Employed']): patch.set_facecolor(PAL[s]); patch.set_alpha(0.8)
ax.set_title('Work Readiness Score vs employment', fontweight='bold')
ax.set_ylabel('Work Readiness Score (0-1)'); plt.tight_layout(); plt.show()


**What it means:** heavily overlapping boxes mean the score separates weakly — and adding weak features to a model can even *hurt* it. Finding which combinations genuinely help is part of the challenge.

## 🏗️ Step 3: A first baseline model
We start with **logistic regression**, but deliberately choose two features that carry little signal:
`work_readiness_score` and `sa_citizen`. This sets a low floor and shows that adding variables is not enough on its own.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

tr = train_data[train_data['employed_status'].notna()].copy()
y = tr['employed_status'].astype(int)

def cat_pipe():
    return Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
                     ('oh', OneHotEncoder(handle_unknown='ignore'))])
def num_pipe():
    return Pipeline([('impute', SimpleImputer(strategy='median'))])

# Baseline: two weak features
base_cat = ['sa_citizen']
base_num = ['work_readiness_score']
Xtr = tr[base_cat + base_num].copy(); Xte = test_data[base_cat + base_num].copy()
for df in (Xtr, Xte):
    df['sa_citizen'] = df['sa_citizen'].astype(str)

pre = ColumnTransformer([('cat', cat_pipe(), base_cat), ('num', num_pipe(), base_num)])
baseline = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
baseline.fit(Xtr, y)
print('Baseline trained on:', base_cat + base_num)


## 🔮 Step 4: Predict and build a submission
For AUC we output **probabilities**, not hard 0/1 labels. The cell below writes `submission_baseline.csv` in the
correct format — upload it on the competition's Submit page to see your score on the public leaderboard.

In [ ]:
test_prob = baseline.predict_proba(Xte)[:, 1]
submission = pd.DataFrame({'anonymised_id': test_data['anonymised_id'], 'employed_status': test_prob})
submission.to_csv('submission_baseline.csv', index=False)
print('Saved submission_baseline.csv', submission.shape)
submission.head()


## 🔄 Step 5: Choose better features and watch the score improve
The baseline features barely helped. Now we swap in features that actually carry signal — `gender` and
`education_level` — while keeping `work_readiness_score`. Same model, better inputs, better score.
The lesson: **which** variables you pick matters more than how many.

In [ ]:
up_cat = ['gender', 'education_level']
up_num = ['work_readiness_score']
Xtr2 = tr[up_cat + up_num].copy(); Xte2 = test_data[up_cat + up_num].copy()
for df in (Xtr2, Xte2):
    df['gender'] = df['gender'].fillna('Missing').astype(str)
    df['education_level'] = df['education_level'].fillna('Missing').astype(str)

pre2 = ColumnTransformer([('cat', cat_pipe(), up_cat), ('num', num_pipe(), up_num)])
upgraded = Pipeline([('pre', pre2), ('clf', LogisticRegression(max_iter=1000))])
upgraded.fit(Xtr2, y)

up_prob = upgraded.predict_proba(Xte2)[:, 1]
submission2 = pd.DataFrame({'anonymised_id': test_data['anonymised_id'], 'employed_status': up_prob})
submission2.to_csv('submission_upgraded.csv', index=False)
print('Saved submission_upgraded.csv', submission2.shape)
submission2.head()


## 🚀 Over to you — ideas to climb the leaderboard
You just saw that swapping weak features for better ones lifted the score. There is much more room to improve:

- **Use participant history:** `employed_lag`, `status_lag`, `status_broad_lag`, `tenure_lag` — a person's *previous* status is a strong clue when present.
- **Handle missingness thoughtfully:** treat 'missing' as its own category; train and test differ in how complete some columns are.
- **More demographics & geography:** `age`, `race`, `province`, `district`, `municipality`.
- **Education & skills:** `school_quintile`, the `matric_*` subjects.
- **Try stronger models:** random forests or gradient boosting (LightGBM / XGBoost).
- **Always submit probabilities** and validate with AUC.

Good luck!